# Part IV — Systems of Equations

## Trefethen & Bau, *Numerical Linear Algebra* (1997) — Lecture 20–23

这是《Numerical Linear Algebra》读书笔记的第 4 册。目标不是摘要原书，而是把每一讲整理成一份**可以独立读懂的数值线性代数讲义**，再把它映射到现代 ML systems。

每一讲尽量保持同一结构：数学对象 → 关键公式 → 几何/算法解释 → numerical stability → ML systems mapping → Python experiment。

本册自洽：下面的 setup cell 提供全部依赖，按顺序 run all 即可。

原书 PDF：https://www.stat.uchicago.edu/~lekheng/courses/309/books/Trefethen-Bau.pdf

---

**本系列共 6 册**（Trefethen & Bau, *Numerical Linear Algebra*, 40 Lectures）

| | |
|---|---|
| Part I | [Fundamentals](01_fundamentals.ipynb) |
| Part II | [QR Factorization and Least Squares](02_qr_least_squares.ipynb) |
| Part III | [Conditioning and Stability](03_conditioning_stability.ipynb) |
| Part IV | [Systems of Equations](04_systems_of_equations.ipynb) |
| Part V | [Eigenvalues](05_eigenvalues.ipynb) |
| Part VI | [Iterative Methods](06_iterative_methods.ipynb) |

索引与阅读顺序见 [00_index.ipynb](00_index.ipynb)。


In [1]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import scipy.linalg as sla
    import scipy.sparse.linalg as spla
    SCIPY=True
except Exception:
    SCIPY=False
rng=np.random.default_rng(7)
np.set_printoptions(precision=5,suppress=True)

def relerr(a,b):
    return np.linalg.norm(a-b)/max(np.linalg.norm(b),1e-30)

def stable_rank(A):
    s=np.linalg.svd(A,compute_uv=False)
    return np.sum(s*s)/(s[0]*s[0])

def spectral_norm_power(A,steps=30,seed=0):
    r=np.random.default_rng(seed)
    v=r.normal(size=A.shape[1])
    v/=np.linalg.norm(v)
    for _ in range(steps):
        v=A.T@(A@v)
        v/=np.linalg.norm(v)
    return np.linalg.norm(A@v)

def make_cond(n,kappa,seed=0):
    r=np.random.default_rng(seed)
    Q1,_=np.linalg.qr(r.normal(size=(n,n)))
    Q2,_=np.linalg.qr(r.normal(size=(n,n)))
    s=np.geomspace(1,1/kappa,n)
    return Q1@np.diag(s)@Q2.T

print(f'NumPy {np.__version__} | SciPy {SCIPY}')

NumPy 2.4.2 | SciPy True


## Notation / 贯穿全书的符号

我们主要讨论实矩阵；复数情形把转置 $A^T$ 换成共轭转置 $A^*$。

- $A\in\mathbb{R}^{m\times n}$
- 向量 2-norm：

$$
\Vert x\Vert_2=\sqrt{x^Tx}
$$

- induced matrix 2-norm：

$$
\Vert A\Vert_2=\max_{x\neq0}\frac{\Vert Ax\Vert_2}{\Vert x\Vert_2}=\sigma_{\max}(A)
$$

- Frobenius norm：

$$
\Vert A\Vert_F^2=\sum_{ij}a_{ij}^2=\sum_i\sigma_i^2
$$

- condition number：

$$
\kappa_2(A)=\Vert A\Vert_2\Vert A^{-1}\Vert_2
=\frac{\sigma_{\max}}{\sigma_{\min}}
$$

- unit roundoff：记为 $u$。典型 floating-point model：

$$
\mathrm{fl}(a\circ b)=(a\circ b)(1+\delta),\qquad |\delta|\lesssim u.
$$

计算机算一次加减乘除，不会得到精确的 $a\circ b$，而是得到一个带相对误差的结果。逐项：

- $\mathrm{fl}(\cdot)$：floating-point，机器实际算出来的数；
- $a\circ b$：一次精确运算（$\circ$ 是 $+,-,\times,/$）；
- $\delta$：这次运算引入的相对误差；
- $u$：unit roundoff，这种格式「一次正确舍入」的相对误差上限。

左边是机器结果，右边是「真值再乘 $1+\delta$」。约束的是**相对误差**，不是绝对误差：真值若是 $1.0$，FP32 下次运算大约落在 $1\pm 6\times 10^{-8}$，不会无缘无故错到 $1.01$。

常见 $u$：

- FP32：$u\approx 2^{-24}\approx 6\times 10^{-8}$
- FP16：$u\approx 2^{-11}\approx 5\times 10^{-4}$
- BF16：$u\approx 2^{-8}\approx 4\times 10^{-3}$

一次运算只错 $u$。病态问题可能把这个 $u$ 放大成 $\kappa u$ 量级的解误差。所以不要把「格式很粗」「矩阵很病态」「算法多放大了 rounding」三件事混成一句“数值不稳”。

### 区分

不要把下面三个问题混在一起：

1. **operator amplification**：$\Vert A\Vert$ 大不大？
2. **problem conditioning**：$A^{-1}$ 是否敏感？
3. **algorithm stability**：实现是否额外放大 rounding error？

现代 ML numerics 中，大量争论其实是把这三个层次混在了一起。

# Lecture 20 — Gaussian Elimination

### 1. Gaussian elimination = 构造 LU

通过 elementary elimination matrix $L_k$ 消去 column $k$ 下方元素：

$$
L_{n-1}\cdots L_2L_1A=U.
$$

整理得到

$$
A=LU.
$$

然后 solve 分两步：

$$
Ly=b,
\qquad
Ux=y.
$$

### 2. Elimination multiplier

消去 $a_{ik}$ 时

$$
\ell_{ik}=\frac{a_{ik}}{a_{kk}}.
$$

如果 pivot $a_{kk}$ 很小，$|\ell_{ik}|$ 会巨大，产生巨大的 intermediate entries。

### 3. Computational cost

dense factorization 约 $\frac23n^3$ FLOPs；随后每个 RHS 的 triangular solves 约 $O(n^2)$。所以多个 RHS 时 factorization 可以复用。

### 4. ML mapping

小规模 implicit solve、constraint solve、quadratic refinement 都可能受益于 factorization reuse。但如果 matrix 随每个 sample/step 改变，factorization cost 可能主导。

**ML numerics 自测**

**Q1.** GE 在构造什么分解？

**A.** $L_{n-1}\cdots L_1A=U$，即 $A=LU$。然后 $Ly=b$、$Ux=y$。

**Q2.** 最危险的 intermediate 是什么？

**A.** multiplier $\ell_{ik}=a_{ik}/a_{kk}$。pivot $a_{kk}$ 很小，$|\ell|$ 爆炸，中间元素暴涨。

**Q3.** 为什么多个 RHS 时 factorization 值得复用？

**A.** dense 分解约 $\frac23 n^3$，每个 RHS 的三角求解只 $O(n^2)$。矩阵不变时摊还很划算。

**Q4.** ML 里矩阵每步都变，还该不该 factorize？

**A.** 若 $A$ 随 sample/step 变，分解成本可能主导。小而固定的 implicit / constraint solve 才适合复用 $LU$。


# Lecture 21 — Pivoting

### 1. Partial pivoting

在第 $k$ 步，从当前 column 的剩余 rows 中选

$$
p=\arg\max_{i\ge k}|a_{ik}|,
$$

把 row $p$ 与 row $k$ 交换，再做 elimination。

这样 multiplier 满足

$$
|\ell_{ik}|\le1
$$

（对 partial pivoting 的该列）。

### 2. 为什么 pivoting 是 numerical algorithm 的一部分

Symbolic algebra 只要求 pivot 非零；floating point 要求 pivot 不能“小到让中间量爆掉”。

### 3. Growth factor

常定义

$$
\rho=\frac{\max_{i,j,k}|a_{ij}^{(k)}|}{\max_{i,j}|a_{ij}|}.
$$

如果 elimination 中元素增长巨大，rounding error 会被相应放大。

### 4. Deployment mapping

自己写/编译 linear solver 时，不能只比较最终 equation 是否能解；还要测试 adversarial scaling 和 pivot pattern。

In [17]:
A=np.array([[1e-12,1.],[1.,1.]])
b=np.array([1.,2.])
A0=A.copy()
b0=b.copy()
m=A0[1,0]/A0[0,0]
A0[1]-=m*A0[0]
b0[1]-=m*b0[0]
x2=b0[1]/A0[1,1]
x1=(b0[0]-A0[0,1]*x2)/A0[0,0]
print(f'no-pivot multiplier {m} naive {np.array([x1, x2])} pivoted {np.linalg.solve(A, b)}')

no-pivot multiplier 1000000000000.0 naive [0.99998 1.     ] pivoted [1. 1.]


**ML numerics 自测**

**Q1.** partial pivoting 在防什么？

**A.** 第 $k$ 步选 $|a_{ik}|$ 最大的行交换，使 $|\ell_{ik}|\le 1$。symbolic 只要求 pivot 非零；浮点要求它不能小到让中间量爆掉。

**Q2.** growth factor $\rho$ 在量什么？

**A.** elimination 过程中元素相对原矩阵涨了多少。$\rho$ 大，rounding 就被同比放大。

**Q3.** 自己写/编译 solver 时只测“能解”够吗？

**A.** 不够。还要测 adversarial scaling 和 pivot pattern，不只是随机可解矩阵。

**Q4.** 该把 pivoting 看成实现细节还是算法的一部分？

**A.** 算法的一部分。没有它，GE 在浮点下可以完全不可用。


# Lecture 22 — Stability of Gaussian Elimination

### 1. GEPP 的稳定性框架

Gaussian elimination with partial pivoting (GEPP) 的 backward error 大致可写成

$$
(A+\Delta A)\hat x=b,
$$

其中

$$
\frac{\Vert \Delta A\Vert}{\Vert A\Vert}
\lesssim C(n)\rho u.
$$

这里 $\rho$ 是 growth factor。

### 2. 这说明什么

- 如果 $\rho$ 温和，GEPP 通常非常可靠；
- 最坏情况下 $\rho$ 可以很大；
- 实际随机/工程矩阵中最坏情形很少见，但不能因此忽视 structured adversarial case。

### 3. Testing methodology

一个真正的 solver parity harness 应扫：

- $\kappa(A)$；
- row/column scale range；
- pivot magnitude；
- growth factor；
- residual；
- FP64 forward reference。

随机 Gaussian matrix 只能覆盖很小一部分数值空间。

**ML numerics 自测**

**Q1.** GEPP 的 backward error 长什么样？

**A.** $(A+\Delta A)\hat x=b$，且 $\Vert\Delta A\Vert/\Vert A\Vert\lesssim C(n)\rho u$。稳定性被 growth factor $\rho$ 卡住。

**Q2.** $\rho$ 温和 / 最坏 / 工程矩阵分别意味着什么？

**A.** 温和则 GEPP 通常很可靠；最坏 $\rho$ 可以很大；随机/工程矩阵很少碰到最坏，但不能忽略 structured adversarial case。

**Q3.** solver parity harness 至少该扫哪些量？

**A.** $\kappa(A)$、行列 scale、pivot 大小、growth factor、residual、相对 FP64 的 forward error。

**Q4.** 为什么随机 Gaussian 测不够？

**A.** 它只覆盖很小一块数值空间，漏掉 scale、亏秩、adversarial pivot 这些会让编译后 solver 翻车的情形。


# Lecture 23 — Cholesky Factorization

### 1. Cholesky 的数学条件

若 $A$ symmetric positive definite (SPD)：

$$
x^TAx>0\quad\forall x\neq0,
$$

则存在唯一 lower triangular $L$（positive diagonal）使

$$
A=LL^T.
$$

### 2. Scalar recurrence

第一列：

$$
l_{11}=\sqrt{a_{11}},
\qquad
l_{i1}=a_{i1}/l_{11}.
$$

一般地，

$$
l_{jj}=\sqrt{a_{jj}-\sum_{k<j}l_{jk}^2},
$$

$$
l_{ij}=\frac{a_{ij}-\sum_{k<j}l_{ik}l_{jk}}{l_{jj}}.
$$

所以最危险的量就是 square-root 里面的 pivot：

$$
p_j=a_{jj}-\sum_{k<j}l_{jk}^2.
$$

理论 SPD 要求 $p_j>0$。

### 3. 为什么 low precision 会让 Cholesky fail

如果最小 eigenvalue 很小，rounding perturbation $E$ 可能满足

$$
\Vert E\Vert_2\gtrsim\lambda_{\min}(A).
$$

那么 $A+E$ 可以变成 indefinite，即使原始 $A$ 数学上 SPD。

### 4. Damping

$$
A_\lambda=A+\lambda I
$$

把 eigenvalues 全部平移：

$$
\lambda_i(A_\lambda)=\lambda_i(A)+\lambda.
$$

因此

$$
\kappa(A_\lambda)
=\frac{\lambda_{\max}+\lambda}{\lambda_{\min}+\lambda}.
$$

这解释了 damping 为什么同时改善数值稳定性和 optimization geometry。

### 5. ML mapping

Hessian/Gauss–Newton/trajectory-refinement 中，小矩阵完全可以让大部分 pipeline 保持 BF16，而把 Gram accumulation + Cholesky + triangular solve 留在 FP32。

In [18]:
n=30
Q,_=np.linalg.qr(rng.normal(size=(n,n)))
ev=np.geomspace(1,1e-10,n)
H=Q@np.diag(ev)@Q.T
print(f'cond(H) {np.linalg.cond(H)} min eig {np.linalg.eigvalsh(H)[0]}')
Hrt=H.astype(np.float16).astype(np.float32)
print(f'min eig after FP16 roundtrip {np.linalg.eigvalsh(Hrt)[0]}')
try:
    np.linalg.cholesky(Hrt)
    print(f'Cholesky after FP16 roundtrip succeeded')
except np.linalg.LinAlgError: print(f'Cholesky after FP16 roundtrip FAILED')
for lam in [0,1e-8,1e-6,1e-4,1e-2]:
    Hd=H+lam*np.eye(n)
    print(f'lambda {lam} cond {np.linalg.cond(Hd)} min eig {np.linalg.eigvalsh(Hd)[0]}')

cond(H) 9999999844.30773 min eig 1.0000000048860106e-10
min eig after FP16 roundtrip -7.172829e-05
Cholesky after FP16 roundtrip FAILED
lambda 0 cond 9999999844.30773 min eig 1.0000000048860106e-10
lambda 1e-08 cond 99009901.94059847 min eig 1.01000000000951e-08
lambda 1e-06 cond 999901.0099048485 min eig 1.0000999999987644e-06
lambda 0.0001 cond 10000.989999009751 min eig 0.00010000010000000686
lambda 0.01 cond 100.99999898999995 min eig 0.010000000099999996


**ML numerics 自测**

**Q1.** Cholesky 需要什么数学条件？

**A.** $A$ SPD：$x^TAx>0$。于是 $A=LL^T$，$L$ 下三角且对角为正。

**Q2.** 最危险的中间量是哪个？

**A.** pivot $p_j=a_{jj}-\sum_{k<j}l_{jk}^2$，再开方。理论 SPD 要求 $p_j>0$；rounding 可以把它打成负的。

**Q3.** 低精度为什么会让数学上 SPD 的 $A$ 分解失败？

**A.** 若 $\Vert E\Vert_2\gtrsim\lambda_{\min}(A)$，则 $A+E$ 可以变成 indefinite。

**Q4.** damping $A+\lambda I$ 同时改善了哪两件事？

**A.** $\lambda_i\leftarrow\lambda_i+\lambda$，于是 $\kappa=(\lambda_{\max}+\lambda)/(\lambda_{\min}+\lambda)$。既抬 pivot，也限制弱曲率方向的 inverse gain。Hessian/GN 里 Gram+Cholesky+solve 常该留在 FP32。
